# Notebook de Baselines — Telco Customer Churn
> **Responsável:** Registro de experimentos no MLflow (parâmetros, métricas, versão do dataset)

---

In [23]:
import subprocess
import sys
import os

print(f"Python versão: {sys.version}")
print(f"Diretório atual: {os.getcwd()}")

# Monta o caminho correto dinamicamente
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)  # sobe UMA pasta (src/ → raiz)
req_path = os.path.join(PROJECT_ROOT, 'requirements.txt')

print(f"Buscando requirements em: {req_path}")

if not os.path.exists(req_path):
    print("requirements.txt não encontrado! Verifique o caminho.")
else:
    print("Verificando dependências...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", req_path])
        print("Dependências prontas! =)")
    except Exception as e:
        print(f"Erro ao instalar: {e}")
        raise

Python versão: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
Diretório atual: c:\Users\cecil\Tech-challenge-step-1\Tech-challenge-step-1\src
Buscando requirements em: c:\Users\cecil\Tech-challenge-step-1\Tech-challenge-step-1\requirements.txt
Verificando dependências...
Dependências prontas! =)


In [24]:
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# MLflow: biblioteca principal
import mlflow
import mlflow.sklearn

# Modelos de baseline do Scikit-learn
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Pré-processamento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print('Todas as libs importadas com sucesso! =)')

Todas as libs importadas com sucesso! =)


In [25]:
# Configuração do MLflow com SQLite

import os
import mlflow

# Descobre o caminho da pasta do notebook automaticamente
NOTEBOOK_DIR = os.path.abspath('')

# Sobe uma pasta (notebooks/ → raiz do projeto)
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)

# Muda para a raiz do projeto para garantir escalabilidade local em qualquer máquina
os.chdir(PROJECT_ROOT)

print('📁 Raiz do projeto:', PROJECT_ROOT)
print('📁 Pasta atual:', os.getcwd())

# Configura o MLflow na raiz do projeto
mlflow.set_tracking_uri('sqlite:///mlruns.db')
EXPERIMENT_NAME = 'telco-churn-baselines'
mlflow.set_experiment(EXPERIMENT_NAME)

print('✅ Configurado! mlruns.db salvo em:', os.path.join(PROJECT_ROOT, 'mlruns.db'))

📁 Raiz do projeto: c:\Users\cecil\Tech-challenge-step-1\Tech-challenge-step-1
📁 Pasta atual: c:\Users\cecil\Tech-challenge-step-1\Tech-challenge-step-1
✅ Configurado! mlruns.db salvo em: c:\Users\cecil\Tech-challenge-step-1\Tech-challenge-step-1\mlruns.db


---
## Carregamento e Limpeza do Dataset

In [26]:
from pathlib import Path

csv_path = Path.home() / "Tech-challenge-step-1" / "Tech-challenge-step-1" / "src" / "data" / "churn.csv"

df = pd.read_csv(csv_path)

print(f'Shape: {df.shape}')   # linhas x colunas
print(f'Colunas: {df.columns.tolist()}')
df.head()

Shape: (7043, 21)
Colunas: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [27]:
# Limpeza básica necessária para este dataset

# 1. Remove customerID — identificador único, não é uma feature preditiva
df = df.drop(columns=['customerID'])

# 2. TotalCharges vem como string (tem espaços em branco) — converte para float
#    Linhas com valores inválidos viram NaN e são removidas
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# 3. Converte a coluna alvo Churn de Yes/No para 1/0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'Shape após limpeza: {df.shape}')
print(f'\nDistribuição do alvo (Churn):')
print(df['Churn'].value_counts(normalize=True).round(3))

Shape após limpeza: (7032, 20)

Distribuição do alvo (Churn):
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64


---
## Pré-processamento para os Modelos

In [28]:
# Codificação das variáveis categóricas (Label Encoding), como modelos de ML precisam de números aqui nós convertemos texto em 0/1/2...

df_model = df.copy()  # preserva o df original intacto

# Identifica colunas categóricas (tipo object = texto)
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'Colunas categóricas: {cat_cols}')

# Aplica Label Encoding em cada uma
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

print('\n✅ Encoding aplicado. Primeiras linhas:')
df_model.head(3)

Colunas categóricas: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

✅ Encoding aplicado. Primeiras linhas:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1


In [29]:
# Split treino/teste
# X = features (todas as colunas menos o alvo)
# y = alvo (Churn: 0 ou 1)
# test_size=0.2 → 80% treino, 20% teste
# random_state=42 → garante que o split seja sempre o mesmo (reprodutibilidade)
# stratify=y → mantém a proporção de churn/não-churn em treino e teste

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # importante usarmos esse aqui pra dataset desbalanceado!
)

print(f'Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas')
print(f'Proporção churn no treino: {y_train.mean():.2%}')
print(f'Proporção churn no teste:  {y_test.mean():.2%}')

Treino: 5625 linhas | Teste: 1407 linhas
Proporção churn no treino: 26.58%
Proporção churn no teste:  26.58%


In [30]:
# Registra o dataset no MLflow (versão)
# mlflow.data.from_pandas cria um objeto que documenta:
# - de onde veio o arquivo (source)
# - qual coluna é o alvo (targets)
# - um hash SHA256 gerado automaticamente — funciona como 'versão'

dataset_mlflow = mlflow.data.from_pandas(
    df,                          # dataframe ANTES do encoding (mais legível)
    source='../src/data/churn.csv',  # caminho do arquivo
    name='telco_churn_v1',           # nome da versão
    targets='Churn'                  # coluna alvo
)

print(f' Dataset registrado: {dataset_mlflow.name}')
print(f'   Linhas: {len(df)} | Colunas: {len(df.columns)}')
print(f'   Hash (versão automática): {dataset_mlflow.digest}')

 Dataset registrado: telco_churn_v1
   Linhas: 7032 | Colunas: 20
   Hash (versão automática): af7c4e29


---
## Registro dos Baselines no MLflow


In [31]:
# Baseline 1: DummyClassifier
# O DummyClassifier é o "piso" de performance: ele não aprende nada,só chuta a classe mais frequente. Serve para ter uma referência mínima.
# Se qualquer modelo real não bater o Dummy, algo está errado.

with mlflow.start_run(run_name='baseline_dummy'):

    print('1️⃣ Iniciando run...')

    params = {'strategy': 'most_frequent', 'random_state': 42}
    mlflow.log_params(params)
    print('2️⃣ Parâmetros logados!')

    dummy = DummyClassifier(**params)
    dummy.fit(X_train, y_train)
    y_pred = dummy.predict(X_test)
    print('3️⃣ Modelo treinado!')

    mlflow.log_metrics({
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc':  roc_auc_score(y_test, y_pred)
    })
    print('4️⃣ Métricas logadas!')

    mlflow.log_params({
        'dataset_nome': 'telco_churn_v1',
        'dataset_arquivo': '../src/data/churn.csv',
        'dataset_linhas': len(df),
        'dataset_colunas': len(df.columns)
    })
    print('5️⃣ Versão do dataset logada!')

    mlflow.set_tags({'modelo_tipo': 'baseline', 'responsavel': 'Cecilia'})
    print('6️⃣ Tags salvas!')

    print('Tudo registrado! =)')

1️⃣ Iniciando run...
2️⃣ Parâmetros logados!
3️⃣ Modelo treinado!
4️⃣ Métricas logadas!
5️⃣ Versão do dataset logada!
6️⃣ Tags salvas!
Tudo registrado! =)


In [32]:
# Baseline 2: Regressão Logística
# A Regressão Logística é um modelo linear simples, muito usado como baseline
# Usamos um Pipeline para normalizar os dados antes de treinar

with mlflow.start_run(run_name='baseline_regressao_logistica'):

    print('1️⃣ Iniciando run...')

    params = {
        'C': 1.0,
        'max_iter': 1000,
        'solver': 'lbfgs',
        'random_state': 42
    }
    mlflow.log_params(params)
    mlflow.log_param('preprocessamento', 'StandardScaler')
    print('2️⃣ Parâmetros logados!')

    pipeline_lr = Pipeline([
        ('scaler', StandardScaler()), # StandardScaler deixa todas as features na mesma escala
        ('modelo', LogisticRegression(**params))
    ])
    pipeline_lr.fit(X_train, y_train)
    y_pred  = pipeline_lr.predict(X_test)
    y_proba = pipeline_lr.predict_proba(X_test)[:, 1]
    print('3️⃣ Modelo treinado!')

    mlflow.log_metrics({
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc':  roc_auc_score(y_test, y_proba)
    })
    print('4️⃣ Métricas logadas!')

    mlflow.log_params({
        'dataset_nome': 'telco_churn_v1',
        'dataset_arquivo': '../src/data/churn.csv',
        'dataset_linhas': len(df),
        'dataset_colunas': len(df.columns)
    })
    print('5️⃣ Versão do dataset logada!')

    mlflow.set_tags({
        'modelo_tipo': 'baseline',
        'dataset_versao': 'v1',
        'responsavel': 'Cecilia'
    })
    print('6️⃣ Tags salvas!')

    print('Regressão Logística registrada! =)')
    print(classification_report(y_test, y_pred, target_names=['Não Churn', 'Churn']))

1️⃣ Iniciando run...
2️⃣ Parâmetros logados!
3️⃣ Modelo treinado!
4️⃣ Métricas logadas!
5️⃣ Versão do dataset logada!
6️⃣ Tags salvas!
Regressão Logística registrada! =)
              precision    recall  f1-score   support

   Não Churn       0.85      0.88      0.86      1033
       Churn       0.62      0.56      0.59       374

    accuracy                           0.79      1407
   macro avg       0.74      0.72      0.73      1407
weighted avg       0.79      0.79      0.79      1407



---
## Comparação dos Baselines

In [38]:
# Célula de limpeza — apaga runs duplicadas e mantém só as mais recentes - manter somente em período de aprendizagem do curso
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Busca todas as runs do experimento
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

# Mantém só a run mais recente de cada nome
runs_para_manter = (
    runs.sort_values('start_time', ascending=False)
        .drop_duplicates(subset='tags.mlflow.runName')
        ['run_id']
        .tolist()
)

# Deleta as demais
deletadas = 0
for run_id in runs['run_id']:
    if run_id not in runs_para_manter:
        client.delete_run(run_id)
        deletadas += 1

print(f' {deletadas} runs deletadas! Sobraram {len(runs_para_manter)} runs.')

 0 runs deletadas! Sobraram 6 runs.


In [39]:
# Busca os resultados registrados e exibe uma tabela comparativa
# mlflow.search_runs busca todas as runs do experimento atual

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=['metrics.roc_auc DESC']  # ordena pelo melhor ROC-AUC
)

# Seleciona apenas as colunas relevantes para exibir
cols = ['tags.mlflow.runName', 'metrics.accuracy', 'metrics.f1_score', 'metrics.roc_auc']
tabela = runs[cols].rename(columns={
    'tags.mlflow.runName': 'Modelo',
    'metrics.accuracy': 'Accuracy',
    'metrics.f1_score': 'F1-Score',
    'metrics.roc_auc': 'ROC-AUC'
})

print('=== Comparativo de Baselines ===')
tabela.style.format({'Accuracy': '{:.4f}', 'F1-Score': '{:.4f}', 'ROC-AUC': '{:.4f}'})

=== Comparativo de Baselines ===


,Modelo,Accuracy,F1-Score,ROC-AUC
0,evaluate_lr,nan,0.5927,0.8345
1,baseline_regressao_logistica,0.7939,0.5927,0.8345
2,evaluate_dummy,nan,0.0000,0.5000
3,baseline_dummy,0.7342,0.0000,0.5000
4,log_model_dummy,nan,nan,nan
5,log_model_lr,nan,nan,nan


In [34]:
# Comparação direta de Dummy vs Regressão Logística

dummy_row = runs[runs['tags.mlflow.runName'] == 'baseline_dummy'].iloc[0]
lr_row = runs[runs['tags.mlflow.runName'] == 'baseline_regressao_logistica'].iloc[0]

dummy_metrics = {
    'accuracy': float(dummy_row['metrics.accuracy']),
    'f1_score': float(dummy_row['metrics.f1_score']),
    'roc_auc': float(dummy_row['metrics.roc_auc'])
}
lr_metrics = {
    'accuracy': float(lr_row['metrics.accuracy']),
    'f1_score': float(lr_row['metrics.f1_score']),
    'roc_auc': float(lr_row['metrics.roc_auc'])
}

print('=== Validação Manual dos Resultados ===')
print('DummyClassifier:', dummy_metrics)
print('Regressão Logística:', lr_metrics)
print()

for metric in ['accuracy', 'f1_score', 'roc_auc']:
    diff = lr_metrics[metric] - dummy_metrics[metric]
    winner = 'Regressão Logística' if diff > 0 else 'DummyClassifier' if diff < 0 else 'Empate'
    print(f'{metric}: {dummy_metrics[metric]:.4f} vs {lr_metrics[metric]:.4f} → {winner}')

print('\nComparação concluída. Se quiser usar validate_evaluation_results, atualize o MLflow para uma versão que contenha essa função.')

=== Validação Manual dos Resultados ===
DummyClassifier: {'accuracy': 0.7341862117981521, 'f1_score': 0.0, 'roc_auc': 0.5}
Regressão Logística: {'accuracy': 0.7938877043354655, 'f1_score': 0.5926966292134831, 'roc_auc': 0.8345028498066479}

accuracy: 0.7342 vs 0.7939 → Regressão Logística
f1_score: 0.0000 vs 0.5927 → Regressão Logística
roc_auc: 0.5000 vs 0.8345 → Regressão Logística

Comparação concluída. Se quiser usar validate_evaluation_results, atualize o MLflow para uma versão que contenha essa função.


In [35]:
# Comparação de Dummy vs Regressão Logística utilizando models evaluate

# 1. Loga a Regressão Logística 
with mlflow.start_run(run_name="log_model_lr"):
    mlflow.sklearn.log_model(pipeline_lr, name="model")  # 'name' em vez de 'artifact_path'
    run_id_lr = mlflow.active_run().info.run_id
    model_uri_lr = f"runs:/{run_id_lr}/model"
print("Modelo LR registrado em:", model_uri_lr)

# 2. Loga o DummyClassifier 
with mlflow.start_run(run_name="log_model_dummy"):
    mlflow.sklearn.log_model(dummy, name="model")
    run_id_dummy = mlflow.active_run().info.run_id
    model_uri_dummy = f"runs:/{run_id_dummy}/model"
print("Modelo Dummy registrado em:", model_uri_dummy)

#  3. Prepara o dataframe de avaliação 
eval_df = X_test.copy()
eval_df["Churn"] = y_test.values

#  4. Avalia a Regressão Logística
print("\nAvaliando Regressão Logística...")
with mlflow.start_run(run_name="evaluate_lr"):
    candidate_result = mlflow.models.evaluate(
        model=model_uri_lr,
        data=eval_df,
        targets="Churn",
        model_type="classifier"
        # baseline_model removido no MLflow 3.0
        # comparação manual abaixo
    )

#  5. Avalia o Dummy
print("Avaliando Dummy...")
with mlflow.start_run(run_name="evaluate_dummy"):
    dummy_result = mlflow.models.evaluate(
        model=model_uri_dummy,
        data=eval_df,
        targets="Churn",
        model_type="classifier"
    )

#  6. Comparação manual (substitui o baseline_model) 
print("\n Comparativo LR vs Dummy:")
metricas = ["accuracy_score", "f1_score", "roc_auc"]
for m in metricas:
    lr_val    = candidate_result.metrics.get(m, "N/A")
    dummy_val = dummy_result.metrics.get(m, "N/A")
    if isinstance(lr_val, float) and isinstance(dummy_val, float):
        delta = lr_val - dummy_val
        sinal = "model ok" if delta > 0 else "model notok"
        print(f"  {m:20s}: LR={lr_val:.4f}  Dummy={dummy_val:.4f}  Δ={delta:+.4f} {sinal}")

2026/05/21 20:14:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/21 20:14:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Modelo LR registrado em: runs:/89e87ec154284fe7a51ec9dda006a07c/model


2026/05/21 20:14:55 WARNING mlflow.models.evaluation.evaluators.classifier: According to the evaluation dataset label values, the model type looks like None, but you specified model type 'classifier'. Please verify that you set the `model_type` and `dataset` arguments correctly.
2026/05/21 20:14:55 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as binary dataset, positive label is 1, negative label is 0.


Modelo Dummy registrado em: runs:/7f6ef43de68f4580b7bee390858e413c/model

Avaliando Regressão Logística...


2026/05/21 20:14:55 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2026/05/21 20:14:56 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.
2026/05/21 20:14:56 WARNING mlflow.models.evaluation.evaluators.classifier: According to the evaluation dataset label values, the model type looks like None, but you specified model type 'classifier'. Please verify that you set the `model_type` and `dataset` arguments correctly.
2026/05/21 20:14:56 INFO mlflow.models.evaluation.evaluators.classifier: The evaluation dataset is inferred as binary dataset, positive label is 1, negative label is 0.
2026/05/21 20:14:57 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


Avaliando Dummy...


2026/05/21 20:14:58 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.



 Comparativo LR vs Dummy:
  accuracy_score      : LR=0.7939  Dummy=0.7342  Δ=+0.0597 model ok
  f1_score            : LR=0.5927  Dummy=0.0000  Δ=+0.5927 model ok
  roc_auc             : LR=0.8345  Dummy=0.5000  Δ=+0.3345 model ok


---

## Visualizando os Experimentos no MLflow UI

Após rodar todas as células acima, siga os passos abaixo para visualizar os experimentos registrados:

### Passo 1 — Abra um terminal na raiz do projeto

No VS Code: **Terminal → Novo Terminal**

### Passo 2 — Execute o comando abaixo

```bash
python -m mlflow ui --backend-store-uri sqlite:///mlruns.db
```

### Passo 3 — Acesse no navegador

 http://localhost:5000

---

### Em caso de erro

Se aparecer erro ao rodar o comando acima, use o caminho absoluto do arquivo de banco de dados:

```bash
python -m mlflow ui --backend-store-uri "sqlite:///C:\\Users\\cecil\\Tech-challenge-step-1\\Tech-challenge-step-1\\src\\mlruns.db"
```

> **Atenção:** este caminho absoluto é específico para a minha máquina.
> Se você clonou o repositório em outro computador, substitua pelo caminho completo
> onde o arquivo `mlruns.db` foi gerado na **sua** máquina.
>
> Para descobrir o caminho correto no seu PC, rode no notebook:
> ```python
> import os
> print(os.path.abspath('mlruns.db'))
> ```

---

### O que você vai encontrar no MLflow UI

| Aba | O que tem |
|---|---|
| **Experiments** | Todas as runs registradas com parâmetros e métricas |
| **Models** | Versões dos modelos salvos no Model Registry |
| **Compare** | Selecione duas runs e compare métricas lado a lado |